In [1]:
import os
import re
import json
import joblib
import warnings
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from catboost import CatBoostRegressor

import shap

from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")


# ================================================================
# 1. USER CONFIGURATION
# ================================================================

DATA_PATH = (
    r"D:\2026 Work\My Papers\1-3D-printed fiber-reinforced concrete"
    r"\New Work\Data\Data.csv"
)

MODEL_PATH = (
    r"D:\2026 Work\My Papers\1-3D-printed fiber-reinforced concrete"
    r"\New Work\Models\CatBoost\Models\final_model.joblib"
)

RESULTS_PATH = (
    r"D:\2026 Work\My Papers\1-3D-printed fiber-reinforced concrete"
    r"\New Work\Models\CatBoost\Results\results.joblib"
)

SAVE_DIR = (
    r"D:\2026 Work\My Papers\1-3D-printed fiber-reinforced concrete"
    r"\New Work\SHAP AND LIME"
)

TARGET_COLUMN = "CS (MPa)"

RANDOM_STATE = 42

# Number of test observations used for expensive Partition SHAP
PARTITION_TEST_N = 100

# Number of observations used for standard SHAP
SHAP_TEST_N = None

# Grouped permutation repetitions
GROUP_PERMUTATION_REPEATS = 50

# Individual permutation repetitions
INDIVIDUAL_PERMUTATION_REPEATS = 30

# LIME stability settings
LIME_SEEDS = [42, 52, 62, 72, 82]

LIME_NUM_SAMPLES = [
    3000,
    10000
]

LIME_SAMPLE_AROUND_INSTANCE = [
    False,
    True
]

LIME_NUM_FEATURES = 10

# Scientifically defined concrete strength categories (MPa)
STRENGTH_CATEGORIES = {
    "Low": (0, 30),
    "Medium": (30, 50),
    "High": (50, 70),
    "Ultra-high": (70, 200)
}


# ================================================================
# 2. CREATE OUTPUT DIRECTORIES
# ================================================================

SAVE_DIR = Path(SAVE_DIR)

DIRS = [
    SAVE_DIR,
    SAVE_DIR / "Tables",
    SAVE_DIR / "Plots",
    SAVE_DIR / "SHAP",
    SAVE_DIR / "LIME",
    SAVE_DIR / "Ablation",
    SAVE_DIR / "Correlation",
    SAVE_DIR / "Logs"
]

for directory in DIRS:
    directory.mkdir(parents=True, exist_ok=True)


print("=" * 80)
print("CATBOOST SHAP + LIME REVIEWER VALIDATION (COMPLETE CORRECTED)")
print("=" * 80)
print(f"Output directory: {SAVE_DIR}")
print("=" * 80)


# ================================================================
# 3. LOAD SAVED CATBOOST MODEL AND RESULTS
# ================================================================

print("\nLoading saved CatBoost model...")

final_pipeline = joblib.load(MODEL_PATH)
results = joblib.load(RESULTS_PATH)

print("CatBoost model loaded successfully.")
print("Saved modelling results loaded successfully.")

# Check if the model uses native categorical handling
if hasattr(final_pipeline.named_steps["model"], "get_params"):
    model_params = final_pipeline.named_steps["model"].get_params()
    cat_features = model_params.get("cat_features", None)
    if cat_features is not None and len(cat_features) > 0:
        print(f"\nNOTE: Model uses native CatBoost categorical handling with features: {cat_features}")
    else:
        print("\nNOTE: Model uses numeric-input CatBoost (no cat_features specified).")
        print("      XAI analyses are consistent with this actual model.")


# ================================================================
# 4. LOAD ORIGINAL DATA
# ================================================================

print("\nLoading original dataset...")

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8"
)

df.columns = df.columns.str.strip()

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ================================================================
# 5. REPRODUCE ORIGINAL DATA CLEANING
# ================================================================

y = df[TARGET_COLUMN].copy()

X = df.drop(columns=[TARGET_COLUMN]).copy()

# Remove rows with missing target
valid_idx = ~y.isna()

X = X.loc[valid_idx].copy()
y = y.loc[valid_idx].copy()

# Replace infinite values exactly as in original model code
X = X.replace([np.inf, -np.inf], np.nan)


# ================================================================
# 6. REPRODUCE ORIGINAL FEATURE NAME SANITIZATION
# ================================================================

def sanitize_feature_names(dataframe):

    original_names = dataframe.columns.tolist()

    sanitized = []

    for name in original_names:

        clean = re.sub(
            r'[()\[\]{}<>/\\|;:,.\'"]+',
            '_',
            str(name)
        )

        clean = re.sub(
            r'\s+',
            '_',
            clean
        )

        clean = re.sub(
            r'[^a-zA-Z0-9_]',
            '',
            clean
        )

        if clean and clean[0].isdigit():
            clean = "f_" + clean

        if not clean:
            clean = "feature"

        if clean in sanitized:

            counter = 1

            while f"{clean}_{counter}" in sanitized:
                counter += 1

            clean = f"{clean}_{counter}"

        sanitized.append(clean)

    dataframe = dataframe.copy()

    dataframe.columns = sanitized

    return dataframe


original_feature_names = X.columns.tolist()

X = sanitize_feature_names(X)

feature_names = X.columns.tolist()


# ================================================================
# 7. VERIFY MODEL FEATURES
# ================================================================

saved_features = results.get(
    "feature_names",
    feature_names
)

if feature_names != saved_features:

    print("\nFeature order differs from saved model metadata.")

    missing_features = [
        f for f in saved_features
        if f not in X.columns
    ]

    extra_features = [
        f for f in X.columns
        if f not in saved_features
    ]

    if missing_features:
        raise ValueError(
            f"Missing model features: {missing_features}"
        )

    if extra_features:
        print(
            f"Extra dataset columns will be removed: "
            f"{extra_features}"
        )

    X = X.reindex(columns=saved_features)

    feature_names = saved_features


print("\nFeatures used by CatBoost:")

for feature in feature_names:
    print(f"  {feature}")


# ================================================================
# 8. RECOVER ORIGINAL TRAIN / TEST SPLIT
# ================================================================

train_indices = results["train_indices"]
test_indices = results["test_indices"]

if not set(train_indices).issubset(set(X.index)):
    raise ValueError(
        "Saved training indices do not match the current dataset."
    )

if not set(test_indices).issubset(set(X.index)):
    raise ValueError(
        "Saved testing indices do not match the current dataset."
    )


X_train = X.loc[train_indices].copy()
X_test = X.loc[test_indices].copy()

y_train = y.loc[train_indices].copy()
y_test = y.loc[test_indices].copy()


print("\nOriginal model split recovered:")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")


# ================================================================
# 9. RECOVER THE CATBOOST PIPELINE
# ================================================================

if not hasattr(final_pipeline, "named_steps"):
    raise ValueError(
        "The saved object does not appear to be the original "
        "CatBoost sklearn Pipeline."
    )

if "imputer" not in final_pipeline.named_steps:
    raise ValueError(
        "The saved pipeline does not contain the expected imputer."
    )

if "model" not in final_pipeline.named_steps:
    raise ValueError(
        "The saved pipeline does not contain the expected CatBoost model."
    )


imputer = final_pipeline.named_steps["imputer"]

catboost_model = final_pipeline.named_steps["model"]

# Get the imputation strategy from the fitted imputer
imputation_strategy = imputer.strategy


# ================================================================
# 10. TRANSFORM DATA EXACTLY AS ORIGINAL MODEL
# ================================================================

X_train_imp = pd.DataFrame(
    imputer.transform(X_train),
    columns=feature_names,
    index=X_train.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index
)


# ================================================================
# 11. VERIFY SAVED MODEL PERFORMANCE
# ================================================================

y_train_pred = final_pipeline.predict(X_train)

y_test_pred = final_pipeline.predict(X_test)


def calculate_metrics(y_true, y_pred):

    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        )
    }


train_metrics = calculate_metrics(
    y_train,
    y_train_pred
)

test_metrics = calculate_metrics(
    y_test,
    y_test_pred
)


performance_df = pd.DataFrame([
    {
        "Dataset": "Training",
        **train_metrics
    },
    {
        "Dataset": "Testing",
        **test_metrics
    }
])

performance_df.to_csv(
    SAVE_DIR / "Tables" / "model_performance_verification.csv",
    index=False
)

print("\nModel performance verification:")
print(performance_df)


# ================================================================
# 12. FEATURE DEPENDENCE ANALYSIS
# ================================================================

print("\nCalculating feature dependence...")

pearson_corr = X_train_imp.corr(method="pearson")

spearman_corr = X_train_imp.corr(method="spearman")


pearson_corr.to_csv(
    SAVE_DIR / "Correlation" / "pearson_feature_correlation.csv"
)

spearman_corr.to_csv(
    SAVE_DIR / "Correlation" / "spearman_feature_correlation.csv"
)


# ================================================================
# 13. CORRELATION WITH COMPRESSIVE STRENGTH
# ================================================================

target_corr_records = []

for feature in feature_names:

    pearson_value, pearson_p = pearsonr(
        X_train_imp[feature],
        y_train
    )

    spearman_value, spearman_p = spearmanr(
        X_train_imp[feature],
        y_train
    )

    target_corr_records.append({
        "Feature": feature,
        "Pearson_r": pearson_value,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_value,
        "Spearman_p": spearman_p
    })


target_corr_df = pd.DataFrame(
    target_corr_records
).sort_values(
    "Pearson_r",
    key=lambda x: np.abs(x),
    ascending=False
)

target_corr_df.to_csv(
    SAVE_DIR / "Tables" / "feature_target_correlations.csv",
    index=False
)


# ================================================================
# 14. CORRELATION PAIRS
# ================================================================

correlation_pairs = []

for i in range(len(feature_names)):

    for j in range(i + 1, len(feature_names)):

        f1 = feature_names[i]
        f2 = feature_names[j]

        r = pearson_corr.loc[f1, f2]

        correlation_pairs.append({
            "Feature_1": f1,
            "Feature_2": f2,
            "Pearson_r": r,
            "Absolute_r": abs(r)
        })


correlation_pairs_df = pd.DataFrame(
    correlation_pairs
).sort_values(
    "Absolute_r",
    ascending=False
)

correlation_pairs_df.to_csv(
    SAVE_DIR / "Tables" / "feature_correlation_pairs.csv",
    index=False
)


# ================================================================
# 15. PUBLICATION QUALITY CORRELATION HEATMAP
# ================================================================

plt.figure(figsize=(12, 10))

plt.imshow(
    pearson_corr.values,
    aspect="auto"
)

plt.colorbar(
    label="Pearson correlation coefficient"
)

plt.xticks(
    range(len(feature_names)),
    feature_names,
    rotation=90
)

plt.yticks(
    range(len(feature_names)),
    feature_names
)

for i in range(len(feature_names)):

    for j in range(len(feature_names)):

        value = pearson_corr.iloc[i, j]

        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title(
    "Feature Correlation Matrix",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    SAVE_DIR / "Correlation" / "feature_correlation_heatmap.png",
    dpi=600,
    bbox_inches="tight"
)

plt.close()


# ================================================================
# 16. VIF ANALYSIS
# ================================================================

try:

    from statsmodels.stats.outliers_influence import (
        variance_inflation_factor
    )

    vif_records = []

    X_vif = X_train_imp.copy()

    for i, feature in enumerate(X_vif.columns):

        vif_value = variance_inflation_factor(
            X_vif.values,
            i
        )

        vif_records.append({
            "Feature": feature,
            "VIF": vif_value
        })

    vif_df = pd.DataFrame(
        vif_records
    ).sort_values(
        "VIF",
        ascending=False
    )

    vif_df.to_csv(
        SAVE_DIR / "Tables" / "VIF_analysis.csv",
        index=False
    )

    print("\nVIF analysis completed.")

except ImportError:

    print(
        "\nStatsmodels is not installed. "
        "VIF analysis skipped."
    )


# ================================================================
# 17. DEFINE REVIEWER-RELEVANT CORRELATED GROUPS (FINAL CORRECTED)
# ================================================================

def find_feature_exact(feature_aliases):
    """
    Find feature using exact matching.
    """
    for feature in feature_names:
        f_lower = feature.lower()
        for alias in feature_aliases:
            alias_lower = alias.lower()
            # Exact match or alias is the entire feature name
            if f_lower == alias_lower or f_lower == alias_lower.replace('_', ''):
                return feature
    return None

def find_feature_contains(feature_aliases):
    """
    Fallback: Find feature containing alias (for longer names).
    """
    for feature in feature_names:
        f_lower = feature.lower()
        for alias in feature_aliases:
            alias_lower = alias.lower()
            if alias_lower in f_lower or f_lower in alias_lower:
                return feature
    return None

def find_feature(feature_aliases):
    """
    Try exact match first, then contains match.
    """
    result = find_feature_exact(feature_aliases)
    if result is not None:
        return result
    return find_feature_contains(feature_aliases)


# Find features using corrected matching
water_feature = find_feature(["W", "Water", "Water_Content"])
wb_feature = find_feature(["wtob", "W_B", "W/B", "Water_to_Binder"])
silica_feature = find_feature(["SF", "Silica_Fume", "Silica"])
fiber_diameter = find_feature(["Df", "Fiber_Diameter", "Diameter"])
fiber_length = find_feature(["Lf", "Fiber_Length", "Length"])

# CORRECTED: Aspect Ratio is None - it was excluded from the model
aspect_ratio = None

# CORRECTED: SP = Superplasticizer (not Aspect Ratio)
sp_feature = find_feature(["SP", "Superplasticizer"])

opc_feature = find_feature(["OPC", "Cement"])
hpmc_feature = find_feature(["HPMC", "Hydroxypropyl"])
fa_feature = find_feature(["FA", "Fly_Ash"])
slag_feature = find_feature(["GS", "GGBFS", "Slag"])

print("\n" + "=" * 80)
print("FEATURE MAPPING (FINAL CORRECTED)")
print("=" * 80)
print(f"  W (Water): {water_feature}")
print(f"  wtob (W/B): {wb_feature}")
print(f"  SF (Silica Fume): {silica_feature}")
print(f"  Df (Fiber Diameter): {fiber_diameter}")
print(f"  Lf (Fiber Length): {fiber_length}")
print(f"  AR (Aspect Ratio): {aspect_ratio} (excluded from model)")
print(f"  SP (Superplasticizer): {sp_feature}")
print(f"  OPC (Cement): {opc_feature}")
print(f"  HPMC: {hpmc_feature}")
print(f"  FA: {fa_feature}")
print(f"  Slag: {slag_feature}")


# ================================================================
# CORRECTED GROUPS
# ================================================================

reviewer_groups = {}

# Water_WB_Silica = ['W', 'wtob', 'SF']
water_group = []
if water_feature is not None:
    water_group.append(water_feature)
if wb_feature is not None:
    water_group.append(wb_feature)
if silica_feature is not None:
    water_group.append(silica_feature)

if len(water_group) >= 2:
    reviewer_groups["Water_WB_Silica"] = water_group
    print(f"\nCORRECTED Water_WB_Silica group: {water_group}")

# Fiber_Geometry = ['Df', 'Lf'] (SP removed)
fiber_group = []
if fiber_diameter is not None:
    fiber_group.append(fiber_diameter)
if fiber_length is not None:
    fiber_group.append(fiber_length)

if len(fiber_group) >= 2:
    reviewer_groups["Fiber_Geometry"] = fiber_group
    print(f"CORRECTED Fiber_Geometry group: {fiber_group}")

# Binder_System = ['OPC', 'FA', 'Slag', 'SF']
binder_group = []
if opc_feature is not None:
    binder_group.append(opc_feature)
if fa_feature is not None:
    binder_group.append(fa_feature)
if slag_feature is not None:
    binder_group.append(slag_feature)
if silica_feature is not None:
    binder_group.append(silica_feature)

if len(binder_group) >= 2:
    reviewer_groups["Binder_System"] = list(dict.fromkeys(binder_group))
    print(f"Binder_System group: {reviewer_groups['Binder_System']}")

print("\nReviewer-relevant groups:")

for group_name, features in reviewer_groups.items():
    print(f"  {group_name}: {features}")


with open(
    SAVE_DIR / "Tables" / "reviewer_feature_groups.json",
    "w"
) as f:
    json.dump(reviewer_groups, f, indent=4)


# ================================================================
# 18. STANDARD TREE SHAP
# ================================================================

print("\nCalculating standard TreeSHAP...")

if SHAP_TEST_N is None:
    X_shap = X_test_imp.copy()
else:
    X_shap = X_test_imp.sample(
        n=min(SHAP_TEST_N, len(X_test_imp)),
        random_state=RANDOM_STATE
    )


tree_explainer = shap.TreeExplainer(catboost_model)
tree_explanation = tree_explainer(X_shap)

shap_values = np.asarray(tree_explanation.values)
expected_value = tree_explainer.expected_value

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]


# ================================================================
# 19. VERIFY SHAP ADDITIVITY
# ================================================================

shap_prediction_check = expected_value + np.sum(shap_values, axis=1)
model_prediction_check = catboost_model.predict(X_shap)
additivity_error = shap_prediction_check - model_prediction_check

additivity_df = pd.DataFrame({
    "Sample_ID": X_shap.index,
    "Model_Prediction": model_prediction_check,
    "SHAP_Reconstructed_Prediction": shap_prediction_check,
    "Absolute_Error": np.abs(additivity_error)
})

additivity_df.to_csv(
    SAVE_DIR / "SHAP" / "SHAP_additivity_check.csv",
    index=False
)

print(f"Maximum SHAP reconstruction error: {np.max(np.abs(additivity_error)):.6f}")


# ================================================================
# 20. GLOBAL SHAP IMPORTANCE
# ================================================================

mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
mean_signed_shap = np.mean(shap_values, axis=0)

shap_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_ABS_SHAP": mean_abs_shap,
    "Mean_SIGNED_SHAP": mean_signed_shap
}).sort_values("Mean_ABS_SHAP", ascending=False)

shap_importance_df["SHAP_Rank"] = np.arange(1, len(shap_importance_df) + 1)

shap_importance_df.to_csv(
    SAVE_DIR / "Tables" / "SHAP_global_importance.csv",
    index=False
)


# ================================================================
# 21. SHAP SUMMARY PLOT
# ================================================================

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    max_display=len(feature_names),
    show=False
)
plt.title("SHAP Summary Plot", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "SHAP" / "SHAP_summary_test_set.png", dpi=600, bbox_inches="tight")
plt.close()


# ================================================================
# 22. SHAP BAR PLOT
# ================================================================

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    plot_type="bar",
    max_display=len(feature_names),
    show=False
)
plt.title("Global SHAP Feature Importance", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "SHAP" / "SHAP_global_bar.png", dpi=600, bbox_inches="tight")
plt.close()


# ================================================================
# 23. SHAP DEPENDENCE PLOTS
# ================================================================

dependence_features = []
seen = set()

for feature in [water_feature, wb_feature, silica_feature,
                fiber_diameter, fiber_length,
                opc_feature, hpmc_feature, sp_feature]:
    if feature is not None and feature in feature_names and feature not in seen:
        dependence_features.append(feature)
        seen.add(feature)

print(f"\nGenerating SHAP dependence plots for: {dependence_features}")

for feature in dependence_features:
    print(f"Generating SHAP dependence plot: {feature}")
    
    plt.figure(figsize=(8, 6))
    shap.dependence_plot(
        feature,
        shap_values,
        X_shap,
        feature_names=feature_names,
        interaction_index="auto",
        show=False
    )
    plt.title(f"SHAP Dependence: {feature}", fontsize=15, fontweight="bold")
    plt.tight_layout()
    
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", feature)
    plt.savefig(
        SAVE_DIR / "SHAP" / f"SHAP_dependence_{safe_name}.png",
        dpi=600, bbox_inches="tight"
    )
    plt.close()


# ================================================================
# 24. SHAP INTERACTION VALUES
# ================================================================

print("\nCalculating SHAP interaction values...")

interaction_values = None

try:
    interaction_values = tree_explainer.shap_interaction_values(X_shap)
    interaction_values = np.asarray(interaction_values)
    
    if interaction_values.ndim == 4:
        interaction_values = interaction_values[:, :, :, 0]
    
    mean_abs_interaction = np.mean(np.abs(interaction_values), axis=0)
    
    interaction_df = pd.DataFrame(
        mean_abs_interaction,
        index=feature_names,
        columns=feature_names
    )
    interaction_df.to_csv(SAVE_DIR / "Tables" / "SHAP_interaction_matrix.csv")
    
    # Extract pairwise interactions
    interaction_pairs = []
    for i in range(len(feature_names)):
        for j in range(i + 1, len(feature_names)):
            interaction_pairs.append({
                "Feature_1": feature_names[i],
                "Feature_2": feature_names[j],
                "Mean_ABS_SHAP_Interaction": mean_abs_interaction[i, j]
            })
    
    interaction_pairs_df = pd.DataFrame(interaction_pairs).sort_values(
        "Mean_ABS_SHAP_Interaction", ascending=False
    )
    interaction_pairs_df["Interaction_Rank"] = np.arange(1, len(interaction_pairs_df) + 1)
    interaction_pairs_df.to_csv(SAVE_DIR / "Tables" / "SHAP_top_interactions.csv", index=False)
    
    # Interaction heatmap
    plt.figure(figsize=(12, 10))
    plt.imshow(mean_abs_interaction, aspect="auto")
    plt.colorbar(label="Mean absolute SHAP interaction")
    plt.xticks(range(len(feature_names)), feature_names, rotation=90)
    plt.yticks(range(len(feature_names)), feature_names)
    plt.title("SHAP Interaction Strength", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "SHAP" / "SHAP_interaction_heatmap.png", dpi=600, bbox_inches="tight")
    plt.close()
    
    print("\nTop SHAP interactions:")
    print(interaction_pairs_df.head(15).to_string(index=False))
    
except Exception as e:
    print(f"\nSHAP interaction analysis failed: {str(e)}")


# ================================================================
# 24b. REVIEWER-REQUESTED SHAP INTERACTION PAIRS (FINAL CORRECTED)
# ================================================================

print("\n" + "=" * 80)
print("REVIEWER-REQUESTED INTERACTION PAIRS (FINAL CORRECTED)")
print("=" * 80)

# CORRECTED mapping: Actual feature names
interaction_mapping = {
    "Water": water_feature,
    "W/B": wb_feature,
    "Silica": silica_feature,
    "OPC": opc_feature,  # CORRECTED: Use OPC specifically
    "Fiber_Length": fiber_length,
    "Fiber_Diameter": fiber_diameter,
    "HPMC": hpmc_feature,
    "SP": sp_feature
}

print("\nInteraction mapping used:")
for key, value in interaction_mapping.items():
    print(f"  {key} → {value}")

# CORRECTED interaction pairs (6 pairs, no Aspect Ratio)
# CORRECTED: "OPC × Water" not "Binder × Water"
requested_pairs = [
    ("Water", "W/B"),              # W × wtob
    ("Water", "Silica"),           # W × SF
    ("W/B", "Silica"),             # wtob × SF
    ("OPC", "Water"),              # OPC × W (corrected terminology)
    ("Fiber_Diameter", "HPMC"),    # Df × HPMC
    ("Fiber_Length", "Fiber_Diameter"),  # Lf × Df
]

reviewer_interaction_records = []

print("\nCalculating interaction strengths:")

for pair_name_1, pair_name_2 in requested_pairs:
    f1 = interaction_mapping.get(pair_name_1)
    f2 = interaction_mapping.get(pair_name_2)
    
    if f1 is None or f2 is None:
        print(f"  WARNING: Could not find features for {pair_name_1} × {pair_name_2}")
        continue
    
    if f1 not in feature_names or f2 not in feature_names:
        print(f"  WARNING: {f1} or {f2} not in feature_names")
        continue
    
    i = feature_names.index(f1)
    j = feature_names.index(f2)
    
    if interaction_values is not None:
        mean_interaction = np.mean(np.abs(interaction_values[:, i, j]))
        
        reviewer_interaction_records.append({
            "Interaction_Pair": f"{pair_name_1} × {pair_name_2}",
            "Feature_1": f1,
            "Feature_2": f2,
            "Mean_ABS_SHAP_Interaction": mean_interaction
        })
        
        print(f"  {pair_name_1} × {pair_name_2}: {f1} × {f2} = {mean_interaction:.4f}")

if reviewer_interaction_records:
    reviewer_interaction_df = pd.DataFrame(reviewer_interaction_records)
    reviewer_interaction_df = reviewer_interaction_df.sort_values(
        "Mean_ABS_SHAP_Interaction", ascending=False
    )
    reviewer_interaction_df.to_csv(
        SAVE_DIR / "Tables" / "reviewer_requested_interactions.csv",
        index=False
    )
    
    print("\nReviewer-requested interaction pairs (sorted):")
    print(reviewer_interaction_df.to_string(index=False))


# ================================================================
# 25. CORRELATION-AWARE PARTITION SHAP
# ================================================================

print("\nCalculating correlation-aware Partition SHAP...")

# Note: Test set has 45 observations, so partition_n will be 45
partition_n = min(PARTITION_TEST_N, len(X_test_imp))
X_partition = X_test_imp.sample(n=partition_n, random_state=RANDOM_STATE)

print(f"Partition SHAP using {partition_n} test observations")

try:
    background_n = min(100, len(X_train_imp))
    X_background = X_train_imp.sample(n=background_n, random_state=RANDOM_STATE)
    
    partition_masker = shap.maskers.Partition(X_background, clustering="correlation")
    partition_explainer = shap.Explainer(
        catboost_model.predict,
        partition_masker,
        algorithm="partition",
        feature_names=feature_names
    )
    
    partition_explanation = partition_explainer(X_partition)
    partition_values = np.asarray(partition_explanation.values)
    
    partition_mean_abs = np.mean(np.abs(partition_values), axis=0)
    partition_mean_signed = np.mean(partition_values, axis=0)
    
    partition_df = pd.DataFrame({
        "Feature": feature_names,
        "Partition_SHAP_Mean_ABS": partition_mean_abs,
        "Partition_SHAP_Mean_SIGNED": partition_mean_signed
    }).sort_values("Partition_SHAP_Mean_ABS", ascending=False)
    
    partition_df["Partition_SHAP_Rank"] = np.arange(1, len(partition_df) + 1)
    partition_df.to_csv(SAVE_DIR / "Tables" / "Partition_SHAP_correlation_aware.csv", index=False)
    
    # Compare standard TreeSHAP and Partition SHAP
    comparison_df = shap_importance_df[["Feature", "Mean_ABS_SHAP", "SHAP_Rank"]].merge(
        partition_df[["Feature", "Partition_SHAP_Mean_ABS", "Partition_SHAP_Rank"]],
        on="Feature"
    )
    comparison_df["Absolute_Importance_Ratio"] = (
        comparison_df["Partition_SHAP_Mean_ABS"] /
        comparison_df["Mean_ABS_SHAP"].replace(0, np.nan)
    )
    comparison_df.to_csv(SAVE_DIR / "Tables" / "TreeSHAP_vs_PartitionSHAP.csv", index=False)
    
    print("\nCorrelation-aware Partition SHAP completed.")
    
except Exception as e:
    print(f"\nPartition SHAP failed: {str(e)}")


# ================================================================
# 26. GROUPED SHAP ATTRIBUTION
# ================================================================

grouped_shap_records = []

for group_name, group_features in reviewer_groups.items():
    valid_features = [f for f in group_features if f in feature_names]
    if len(valid_features) == 0:
        continue
    
    feature_indices = [feature_names.index(f) for f in valid_features]
    group_abs_values = np.sum(np.abs(shap_values[:, feature_indices]), axis=1)
    group_signed_values = np.sum(shap_values[:, feature_indices], axis=1)
    
    grouped_shap_records.append({
        "Group": group_name,
        "Features": ", ".join(valid_features),
        "Mean_ABS_Group_SHAP": np.mean(group_abs_values),
        "Mean_SIGNED_Group_SHAP": np.mean(group_signed_values)
    })

grouped_shap_df = pd.DataFrame(grouped_shap_records).sort_values(
    "Mean_ABS_Group_SHAP", ascending=False
)
grouped_shap_df["Group_Rank"] = np.arange(1, len(grouped_shap_df) + 1)
grouped_shap_df.to_csv(SAVE_DIR / "Tables" / "grouped_SHAP_attribution.csv", index=False)


# ================================================================
# 26b. CONDITIONAL STRATIFICATION OF SHAP ATTRIBUTION
# ================================================================

print("\nCalculating conditional stratification of SHAP attribution...")

conditional_stratification_records = []

conditional_features = [
    (water_feature, wb_feature, "Water_WB"),
    (silica_feature, wb_feature, "Silica_WB"),
    (fiber_length, fiber_diameter, "Fiber_Length_Diameter"),
    (opc_feature, water_feature, "OPC_Water")
]

for primary, conditional, group_name in conditional_features:
    if primary is None or conditional is None:
        continue
    if primary not in feature_names or conditional not in feature_names:
        continue
    
    primary_idx = feature_names.index(primary)
    cond_values = X_shap[conditional].values
    
    cond_percentiles = np.percentile(cond_values, [33, 67])
    low_mask = cond_values < cond_percentiles[0]
    high_mask = cond_values > cond_percentiles[1]
    
    if np.sum(low_mask) > 5 and np.sum(high_mask) > 5:
        shap_primary_low = np.mean(np.abs(shap_values[low_mask, primary_idx]))
        shap_primary_high = np.mean(np.abs(shap_values[high_mask, primary_idx]))
        ratio = shap_primary_high / shap_primary_low if shap_primary_low > 0 else np.nan
        
        conditional_stratification_records.append({
            "Group": group_name,
            "Primary_Feature": primary,
            "Conditional_Feature": conditional,
            "Mean_Absolute_SHAP_Low_Conditional": shap_primary_low,
            "Mean_Absolute_SHAP_High_Conditional": shap_primary_high,
            "Attribution_Ratio": ratio,
            "Interpretation": (
                f"The mean absolute SHAP magnitude of {primary} was {ratio:.2f}x "
                f"higher in observations with high {conditional} than in "
                f"observations with low {conditional}"
            )
        })

conditional_stratification_df = pd.DataFrame(conditional_stratification_records)
if len(conditional_stratification_df) > 0:
    conditional_stratification_df.to_csv(
        SAVE_DIR / "Tables" / "conditional_stratification_SHAP_attribution.csv",
        index=False
    )
    print("\nConditional stratification of SHAP attribution:")
    for record in conditional_stratification_records:
        print(f"  {record['Interpretation']}")


# ================================================================
# 27. GROUPED PERMUTATION IMPORTANCE
# ================================================================

print("\nCalculating grouped permutation importance...")

def grouped_permutation_importance(model, X_data, y_data, groups, repeats=30, random_state=42):
    baseline_prediction = model.predict(X_data)
    baseline_r2 = r2_score(y_data, baseline_prediction)
    rng = np.random.default_rng(random_state)
    
    records = []
    for group_name, group_features in groups.items():
        valid_features = [f for f in group_features if f in X_data.columns]
        if len(valid_features) == 0:
            continue
        
        importance_values = []
        for _ in range(repeats):
            permutation = rng.permutation(len(X_data))
            X_permuted = X_data.copy()
            
            for feature in valid_features:
                X_permuted[feature] = X_data[feature].iloc[permutation].to_numpy()
            
            prediction = model.predict(X_permuted)
            permuted_r2 = r2_score(y_data, prediction)
            importance_values.append(baseline_r2 - permuted_r2)
        
        records.append({
            "Group": group_name,
            "Features": ", ".join(valid_features),
            "Baseline_R2": baseline_r2,
            "Mean_R2_Decrease": np.mean(importance_values),
            "SD_R2_Decrease": np.std(importance_values, ddof=1),
            "Median_R2_Decrease": np.median(importance_values),
            "Min_R2_Decrease": np.min(importance_values),
            "Max_R2_Decrease": np.max(importance_values)
        })
    
    return pd.DataFrame(records).sort_values("Mean_R2_Decrease", ascending=False)

grouped_perm_df = grouped_permutation_importance(
    final_pipeline, X_test_imp, y_test, reviewer_groups,
    repeats=GROUP_PERMUTATION_REPEATS, random_state=RANDOM_STATE
)
grouped_perm_df.to_csv(SAVE_DIR / "Tables" / "grouped_permutation_importance.csv", index=False)


# ================================================================
# 28. INDIVIDUAL PERMUTATION IMPORTANCE
# ================================================================

print("\nCalculating individual permutation importance...")

individual_perm = permutation_importance(
    final_pipeline, X_test_imp, y_test,
    n_repeats=INDIVIDUAL_PERMUTATION_REPEATS,
    random_state=RANDOM_STATE, scoring="r2", n_jobs=-1
)

individual_perm_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_R2_Decrease": individual_perm.importances_mean,
    "SD_R2_Decrease": individual_perm.importances_std
}).sort_values("Mean_R2_Decrease", ascending=False)

individual_perm_df["Permutation_Rank"] = np.arange(1, len(individual_perm_df) + 1)
individual_perm_df.to_csv(SAVE_DIR / "Tables" / "individual_permutation_importance.csv", index=False)


# ================================================================
# 29. FEATURE ABLATION / SENSITIVITY ANALYSIS (CORRECTED - Option A)
# ================================================================

print("\nStarting feature-ablation sensitivity analysis...")

def train_ablation_model_original_preprocessing(
    X_train_data, y_train_data, X_test_data, y_test_data,
    dropped_features, cb_params, imputer_strategy="median"
):
    """
    Option A: Use original preprocessing pipeline with imputer + CatBoost.
    This matches the original model's preprocessing workflow.
    """
    # Identify retained features
    retained_features = [f for f in X_train_data.columns if f not in dropped_features]
    
    Xtr = X_train_data[retained_features].copy()
    Xte = X_test_data[retained_features].copy()
    
    # Create pipeline with imputer and CatBoost
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy=imputer_strategy)),
        ("model", CatBoostRegressor(**cb_params, verbose=False))
    ])
    
    # Fit and predict
    pipeline.fit(Xtr, y_train_data)
    prediction = pipeline.predict(Xte)
    
    metrics = calculate_metrics(y_test_data, prediction)
    return metrics, pipeline

cb_params = results.get("best_params", None)

if cb_params is None:
    print("Best CatBoost parameters not found. Ablation skipped.")
else:
    # Remove verbose parameter if present
    cb_params = {k: v for k, v in cb_params.items() if k != "verbose"}
    
    ablation_records = []
    ablation_models = {}
    
    # Full model
    full_metrics = test_metrics
    ablation_records.append({
        "Analysis": "Full model",
        "Dropped_Features": "",
        **full_metrics
    })
    ablation_models["Full model"] = final_pipeline
    
    # Individual feature ablation
    critical_features = [water_feature, wb_feature, silica_feature, 
                        opc_feature, fiber_length, fiber_diameter, sp_feature]
    
    for feature in critical_features:
        if feature is None or feature not in feature_names:
            continue
        
        metrics, pipeline = train_ablation_model_original_preprocessing(
            X_train, y_train, X_test, y_test,
            [feature], cb_params, imputation_strategy
        )
        
        ablation_records.append({
            "Analysis": "Individual feature ablation",
            "Dropped_Features": feature,
            **metrics
        })
        ablation_models[f"Without {feature}"] = pipeline
    
    # Reviewer-specified ablation scenarios
    reviewer_scenarios = [
        ("Without Water", [water_feature] if water_feature else []),
        ("Without W/B", [wb_feature] if wb_feature else []),
        ("Without SF", [silica_feature] if silica_feature else []),
        ("Without Water + W/B", [f for f in [water_feature, wb_feature] if f]),
        ("Without SF + W/B", [f for f in [silica_feature, wb_feature] if f]),
        ("Without fiber variables", [f for f in [fiber_length, fiber_diameter] if f]),
    ]
    
    for scenario_name, drop_features in reviewer_scenarios:
        if len(drop_features) == 0:
            continue
        
        metrics, pipeline = train_ablation_model_original_preprocessing(
            X_train, y_train, X_test, y_test,
            drop_features, cb_params, imputation_strategy
        )
        
        ablation_records.append({
            "Analysis": f"Reviewer scenario: {scenario_name}",
            "Dropped_Features": ", ".join(drop_features),
            **metrics
        })
        ablation_models[scenario_name] = pipeline
    
    # Group ablation
    for group_name, group_features in reviewer_groups.items():
        metrics, pipeline = train_ablation_model_original_preprocessing(
            X_train, y_train, X_test, y_test,
            group_features, cb_params, imputation_strategy
        )
        
        ablation_records.append({
            "Analysis": f"Group ablation: {group_name}",
            "Dropped_Features": ", ".join(group_features),
            **metrics
        })
        ablation_models[f"Group ablation: {group_name}"] = pipeline
    
    ablation_df = pd.DataFrame(ablation_records)
    full_r2, full_rmse, full_mae = full_metrics["R2"], full_metrics["RMSE"], full_metrics["MAE"]
    
    ablation_df["Delta_R2_vs_Full"] = ablation_df["R2"] - full_r2
    ablation_df["Delta_RMSE_vs_Full"] = ablation_df["RMSE"] - full_rmse
    ablation_df["Delta_MAE_vs_Full"] = ablation_df["MAE"] - full_mae
    
    ablation_df.to_csv(SAVE_DIR / "Ablation" / "feature_ablation_sensitivity.csv", index=False)
    
    # Save ablation models
    joblib.dump(ablation_models, SAVE_DIR / "Ablation" / "ablation_models.joblib")
    
    print("\nFeature ablation completed with original preprocessing pipeline.")


# ================================================================
# 30. SELECT REPRESENTATIVE LIME CASES
# ================================================================

print("\nSelecting representative strength cases using scientifically-defined ranges...")

def select_representative_cases(y_values, strength_categories=STRENGTH_CATEGORIES):
    y_array = np.asarray(y_values)
    selected = []
    used = set()
    category_status = {}
    
    for category_name, (low, high) in strength_categories.items():
        candidates = np.where((y_array >= low) & (y_array < high))[0]
        category_status[category_name] = len(candidates)
        
        if len(candidates) == 0:
            print(f"  WARNING: No samples in {category_name} ({low}-{high} MPa)")
            continue
        
        candidate_strengths = y_array[candidates]
        median_idx = candidates[np.argsort(
            np.abs(candidate_strengths - np.median(candidate_strengths))
        )[0]]
        
        if median_idx not in used:
            selected.append((category_name, (low + high) / 2, median_idx))
            used.add(median_idx)
    
    print("\nStrength category availability in test set:")
    for category_name, count in category_status.items():
        status = "AVAILABLE" if count > 0 else "MISSING"
        print(f"  {category_name}: {count} samples - {status}")
    
    return selected

representative_cases = select_representative_cases(y_test.values)

representative_records = []
for category_name, category_median, position in representative_cases:
    sample_id = X_test.index[position]
    actual_strength = y_test.iloc[position]
    predicted_strength = y_test_pred[position]
    
    representative_records.append({
        "Case": category_name,
        "Strength_Category": category_name,
        "Category_Median_MPa": category_median,
        "Sample_ID": sample_id,
        "Actual_CS_MPa": actual_strength,
        "Predicted_CS_MPa": predicted_strength,
        "Absolute_Error_MPa": abs(actual_strength - predicted_strength)
    })

representative_df = pd.DataFrame(representative_records)
representative_df.to_csv(SAVE_DIR / "LIME" / "representative_strength_cases.csv", index=False)

print("\nSelected representative cases:")
print(representative_df.to_string(index=False))

missing_categories = []
for category_name, (low, high) in STRENGTH_CATEGORIES.items():
    if category_name not in representative_df["Case"].values:
        missing_categories.append(f"{category_name} ({low}-{high} MPa)")

if missing_categories:
    print(f"\nNOTE: Missing strength categories in test set:")
    for cat in missing_categories:
        print(f"  - {cat}")


# ================================================================
# 31. GET SHAP VALUES FOR REPRESENTATIVE CASES
# ================================================================

representative_indices = [record[2] for record in representative_cases]
representative_sample_ids = [X_test.index[position] for position in representative_indices]

X_representative = X_test_imp.loc[representative_sample_ids]
representative_shap_exp = tree_explainer(X_representative)
representative_shap_values = np.asarray(representative_shap_exp.values)

if representative_shap_values.ndim == 3:
    representative_shap_values = representative_shap_values[:, :, 0]


# ================================================================
# 32. LIME STABILITY ANALYSIS
# ================================================================

print("\nStarting LIME stability analysis...")

lime_records = []
lime_vectors = {}

for case_number, sample_id in enumerate(representative_sample_ids):
    sample = X_test_imp.loc[sample_id].values.astype(float)
    case_name = representative_df.loc[
        representative_df["Sample_ID"] == sample_id, "Case"
    ].iloc[0]
    
    for seed in LIME_SEEDS:
        for num_samples in LIME_NUM_SAMPLES:
            for sample_around in LIME_SAMPLE_AROUND_INSTANCE:
                explainer = LimeTabularExplainer(
                    training_data=X_train_imp.values,
                    feature_names=feature_names,
                    mode="regression",
                    discretize_continuous=False,
                    sample_around_instance=sample_around,
                    random_state=seed,
                    verbose=False
                )
                
                explanation = explainer.explain_instance(
                    sample, catboost_model.predict,
                    num_features=LIME_NUM_FEATURES,
                    num_samples=num_samples
                )
                
                lime_vector = np.zeros(len(feature_names))
                
                try:
                    maps = explanation.as_map()
                    if len(maps) > 0:
                        label_key = list(maps.keys())[0]
                        for feature_idx, weight in maps[label_key]:
                            lime_vector[feature_idx] = weight
                except Exception:
                    try:
                        for feature_name, weight in explanation.as_list():
                            if feature_name in feature_names:
                                feature_idx = feature_names.index(feature_name)
                                lime_vector[feature_idx] = weight
                    except:
                        pass
                
                for feature_idx, feature in enumerate(feature_names):
                    lime_records.append({
                        "Case": case_name,
                        "Sample_ID": sample_id,
                        "Seed": seed,
                        "Num_Samples": num_samples,
                        "Sample_Around_Instance": sample_around,
                        "Feature": feature,
                        "LIME_Contribution": lime_vector[feature_idx]
                    })
                
                lime_vectors[(case_name, seed, num_samples, sample_around)] = lime_vector

lime_df = pd.DataFrame(lime_records)
lime_df.to_csv(SAVE_DIR / "LIME" / "LIME_all_runs.csv", index=False)

print(f"LIME analysis completed. Total runs: {len(lime_vectors)}")


# ================================================================
# 33. LIME STABILITY METRICS
# ================================================================

stability_records = []

for case_name in representative_df["Case"].tolist():
    case_runs = [(key, vector) for key, vector in lime_vectors.items() if key[0] == case_name]
    vectors = [item[1] for item in case_runs]
    
    pairwise_correlations = []
    for vector_a, vector_b in itertools.combinations(vectors, 2):
        if np.std(vector_a) > 0 and np.std(vector_b) > 0:
            rho, _ = spearmanr(vector_a, vector_b)
            if not np.isnan(rho):
                pairwise_correlations.append(rho)
    
    if len(pairwise_correlations) > 0:
        stability_records.append({
            "Case": case_name,
            "Number_of_LIME_Runs": len(vectors),
            "Mean_Pairwise_Spearman": np.mean(pairwise_correlations),
            "SD_Pairwise_Spearman": np.std(pairwise_correlations, ddof=1),
            "Min_Pairwise_Spearman": np.min(pairwise_correlations),
            "Max_Pairwise_Spearman": np.max(pairwise_correlations)
        })

lime_stability_df = pd.DataFrame(stability_records)
lime_stability_df.to_csv(SAVE_DIR / "LIME" / "LIME_stability_summary.csv", index=False)

print("\nLIME stability metrics calculated.")


# ================================================================
# 34. LIME FEATURE SIGN STABILITY WITH CV AND RANK STABILITY
# ================================================================

print("\nCalculating LIME coefficient of variation and rank stability...")

sign_records = []

for case_name in representative_df["Case"].tolist():
    case_vectors = [vector for key, vector in lime_vectors.items() if key[0] == case_name]
    
    if len(case_vectors) < 2:
        continue
    
    matrix = np.vstack(case_vectors)
    median_vector = np.median(matrix, axis=0)
    
    for feature_idx, feature in enumerate(feature_names):
        feature_values = matrix[:, feature_idx]
        
        reference_sign = np.sign(median_vector[feature_idx])
        if reference_sign == 0:
            sign_stability = np.nan
        else:
            sign_stability = np.mean(np.sign(feature_values) == reference_sign)
        
        mean_val = np.mean(feature_values)
        std_val = np.std(feature_values, ddof=1)
        cv = (std_val / np.abs(mean_val)) if np.abs(mean_val) > 1e-10 else np.nan
        
        # Rank stability: how often feature appears in top 10
        rank_stability_values = []
        for run_idx in range(matrix.shape[0]):
            sorted_indices = np.argsort(np.abs(matrix[run_idx, :]))
            is_in_top10 = feature_idx in sorted_indices[-10:]
            rank_stability_values.append(1.0 if is_in_top10 else 0.0)
        
        rank_stability = np.mean(rank_stability_values)
        
        sign_records.append({
            "Case": case_name,
            "Feature": feature,
            "Median_LIME_Contribution": median_vector[feature_idx],
            "LIME_SD": std_val,
            "Sign_Stability": sign_stability,
            "Coefficient_Variation": cv,
            "Rank_Stability": rank_stability
        })

lime_sign_stability_df = pd.DataFrame(sign_records)
lime_sign_stability_df.to_csv(SAVE_DIR / "LIME" / "LIME_feature_sign_stability.csv", index=False)

# CV summary
cv_summary_records = []
for case_name in lime_sign_stability_df["Case"].unique():
    case_data = lime_sign_stability_df[lime_sign_stability_df["Case"] == case_name]
    
    cv_values = case_data["Coefficient_Variation"].dropna().values
    rank_values = case_data["Rank_Stability"].dropna().values
    sign_values = case_data["Sign_Stability"].dropna().values
    
    cv_summary_records.append({
        "Case": case_name,
        "CV_Mean": np.mean(cv_values) if len(cv_values) > 0 else np.nan,
        "CV_Std": np.std(cv_values, ddof=1) if len(cv_values) > 1 else np.nan,
        "Rank_Stability_Mean": np.mean(rank_values) if len(rank_values) > 0 else np.nan,
        "Rank_Stability_Std": np.std(rank_values, ddof=1) if len(rank_values) > 1 else np.nan,
        "Sign_Stability_Mean": np.mean(sign_values) if len(sign_values) > 0 else np.nan,
        "Sign_Stability_Std": np.std(sign_values, ddof=1) if len(sign_values) > 1 else np.nan,
        "N_Features": len(case_data)
    })

cv_summary_df = pd.DataFrame(cv_summary_records)
cv_summary_df.to_csv(SAVE_DIR / "LIME" / "LIME_cv_rank_stability_summary.csv", index=False)

print("\nLIME coefficient of variation and rank stability calculated.")


# ================================================================
# 35. SHAP VS LIME CONSISTENCY
# ================================================================

print("\nCalculating SHAP-LIME consistency...")

consistency_records = []
global_lime_values = []

for case_number, sample_id in enumerate(representative_sample_ids):
    case_name = representative_df.loc[
        representative_df["Sample_ID"] == sample_id, "Case"
    ].iloc[0]
    
    shap_vector = representative_shap_values[case_number]
    case_lime_runs = [(key, vector) for key, vector in lime_vectors.items() if key[0] == case_name]
    
    for key, lime_vector in case_lime_runs:
        if np.std(shap_vector) > 0 and np.std(lime_vector) > 0:
            spearman_value, _ = spearmanr(shap_vector, lime_vector)
        else:
            spearman_value = np.nan
        
        sign_agreement = np.mean(np.sign(shap_vector) == np.sign(lime_vector))
        
        shap_top_indices = set(np.argsort(np.abs(shap_vector))[-5:])
        lime_top_indices = set(np.argsort(np.abs(lime_vector))[-5:])
        top5_overlap_count = len(shap_top_indices.intersection(lime_top_indices))
        
        union_size = len(shap_top_indices.union(lime_top_indices))
        jaccard = top5_overlap_count / union_size if union_size > 0 else 0.0
        
        consistency_records.append({
            "Case": case_name,
            "Sample_ID": sample_id,
            "Seed": key[1],
            "Num_Samples": key[2],
            "Sample_Around_Instance": key[3],
            "SHAP_LIME_Spearman": spearman_value,
            "Sign_Agreement": sign_agreement,
            "Top5_Overlap_Count": top5_overlap_count,
            "Top5_Jaccard": jaccard
        })
        
        global_lime_values.append(lime_vector)

consistency_df = pd.DataFrame(consistency_records)
consistency_df.to_csv(SAVE_DIR / "LIME" / "SHAP_LIME_consistency_all_runs.csv", index=False)


# ================================================================
# 36. SHAP VS LIME SUMMARY (REVIEWER TABLE FORMAT)
# ================================================================

reviewer_consistency_table = []

for case_name in consistency_df["Case"].unique():
    case_data = consistency_df[consistency_df["Case"] == case_name]
    mean_top5_overlap = case_data["Top5_Overlap_Count"].mean()
    
    reviewer_consistency_table.append({
        "Case": case_name,
        "SHAP-LIME Spearman ρ": case_data["SHAP_LIME_Spearman"].mean(),
        "SD": case_data["SHAP_LIME_Spearman"].std(),
        "Top-5 overlap (count)": mean_top5_overlap,
        "Top-5 Jaccard": case_data["Top5_Jaccard"].mean(),
        "Sign Agreement": case_data["Sign_Agreement"].mean(),
        "N runs": len(case_data)
    })

reviewer_consistency_table_df = pd.DataFrame(reviewer_consistency_table)
reviewer_consistency_table_df = reviewer_consistency_table_df.round(4)

reviewer_consistency_table_df.to_csv(
    SAVE_DIR / "Tables" / "SHAP_LIME_consistency_reviewer_table.csv",
    index=False
)

print("\nSHAP-LIME consistency reviewer table:")
print(reviewer_consistency_table_df.to_string(index=False))


# ================================================================
# 37. FEATURE-LEVEL SHAP VS LIME COMPARISON
# ================================================================

feature_consistency_records = []

for case_number, sample_id in enumerate(representative_sample_ids):
    case_name = representative_df.loc[
        representative_df["Sample_ID"] == sample_id, "Case"
    ].iloc[0]
    
    shap_vector = representative_shap_values[case_number]
    
    case_lime_matrix = np.vstack([
        vector for key, vector in lime_vectors.items() if key[0] == case_name
    ])
    
    mean_lime = np.mean(case_lime_matrix, axis=0)
    sd_lime = np.std(case_lime_matrix, axis=0, ddof=1)
    
    for feature_idx, feature in enumerate(feature_names):
        shap_value = shap_vector[feature_idx]
        lime_value = mean_lime[feature_idx]
        shap_sign = np.sign(shap_value)
        lime_sign = np.sign(lime_value)
        sign_match = shap_sign == lime_sign
        
        feature_consistency_records.append({
            "Case": case_name,
            "Sample_ID": sample_id,
            "Feature": feature,
            "SHAP_Value": shap_value,
            "Mean_LIME_Value": lime_value,
            "SD_LIME_Value": sd_lime[feature_idx],
            "SHAP_Sign": shap_sign,
            "LIME_Sign": lime_sign,
            "Direction_Agreement": sign_match
        })

feature_consistency_df = pd.DataFrame(feature_consistency_records)
feature_consistency_df.to_csv(
    SAVE_DIR / "Tables" / "feature_level_SHAP_LIME_comparison.csv",
    index=False
)


# ================================================================
# 38. IDENTIFY CONTRADICTORY SHAP-LIME DIRECTIONS
# ================================================================

contradictory_df = feature_consistency_df[
    feature_consistency_df["Direction_Agreement"] == False
].copy()

contradictory_df.to_csv(
    SAVE_DIR / "Tables" / "SHAP_LIME_direction_contradictions.csv",
    index=False
)

print(f"\nIdentified {len(contradictory_df)} feature-level direction contradictions.")


# ================================================================
# 38b. DIRECTION VERIFICATION (NON-CAUSAL)
# ================================================================

print("\n" + "=" * 80)
print("DIRECTION VERIFICATION (NON-CAUSAL LANGUAGE)")
print("=" * 80)

direction_records = []

critical_direction_features = [
    water_feature,    # W
    wb_feature,       # wtob
    silica_feature,   # SF
    opc_feature,      # OPC
    fiber_length,     # Lf
    fiber_diameter,   # Df
    sp_feature        # SP
]

# Remove None values
critical_direction_features = [f for f in critical_direction_features if f is not None]

print(f"\nFeatures analyzed for direction: {critical_direction_features}")

for feature in critical_direction_features:
    if feature not in feature_names:
        continue
    
    feature_idx = feature_names.index(feature)
    shap_values_feature = shap_values[:, feature_idx]
    feature_values = X_shap[feature].values
    
    pearson_corr, pearson_p = pearsonr(feature_values, shap_values_feature)
    spearman_corr, spearman_p = spearmanr(feature_values, shap_values_feature)
    
    # Binned mean SHAP (robust direction)
    bins = np.percentile(feature_values, np.linspace(0, 100, 11))
    bin_indices = np.digitize(feature_values, bins)
    binned_means = []
    for i in range(1, len(bins)):
        mask = bin_indices == i
        if np.sum(mask) > 0:
            binned_means.append(np.mean(shap_values_feature[mask]))
        else:
            binned_means.append(np.nan)
    
    valid_means = [m for m in binned_means if not np.isnan(m)]
    if len(valid_means) > 1:
        trend = np.sign(np.mean(np.diff(valid_means)))
    else:
        trend = np.sign(pearson_corr)
    
    if trend > 0:
        direction_text = "positive"
    elif trend < 0:
        direction_text = "negative"
    else:
        direction_text = "neutral"
    
    direction_records.append({
        "Feature": feature,
        "Pearson_r": pearson_corr,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_corr,
        "Spearman_p": spearman_p,
        "Binned_Trend": trend,
        "Interpretation": (
            f"Within the observed dataset, higher values of {feature} "
            f"were associated with {direction_text} "
            f"model contributions (ρ = {spearman_corr:.3f}, p = {spearman_p:.4f})"
        )
    })

direction_df = pd.DataFrame(direction_records)
direction_df.to_csv(SAVE_DIR / "Tables" / "SHAP_direction_verification.csv", index=False)

print("\nDirection verification results:")
for record in direction_records:
    print(f"  {record['Interpretation']}")
print("=" * 80)


# ================================================================
# 39. GLOBAL LIME IMPORTANCE
# ================================================================

if len(global_lime_values) > 0:
    global_lime_matrix = np.vstack(global_lime_values)
    global_lime_importance = np.mean(np.abs(global_lime_matrix), axis=0)
    
    global_lime_df = pd.DataFrame({
        "Feature": feature_names,
        "Mean_ABS_LIME": global_lime_importance
    }).sort_values("Mean_ABS_LIME", ascending=False)
    
    global_lime_df["LIME_Rank"] = np.arange(1, len(global_lime_df) + 1)
    global_lime_df.to_csv(SAVE_DIR / "Tables" / "LIME_global_importance.csv", index=False)
    
    # SHAP vs LIME global ranking
    global_comparison = shap_importance_df[["Feature", "Mean_ABS_SHAP", "SHAP_Rank"]].merge(
        global_lime_df[["Feature", "Mean_ABS_LIME", "LIME_Rank"]],
        on="Feature"
    )
    
    rho, p_value = spearmanr(
        global_comparison["Mean_ABS_SHAP"],
        global_comparison["Mean_ABS_LIME"]
    )
    
    global_comparison["Global_SHAP_LIME_Spearman"] = rho
    global_comparison["Global_SHAP_LIME_p"] = p_value
    global_comparison.to_csv(SAVE_DIR / "Tables" / "global_SHAP_vs_LIME_importance.csv", index=False)


# ================================================================
# 40. REPRESENTATIVE CASE SHAP + LIME TABLES
# ================================================================

for case_number, sample_id in enumerate(representative_sample_ids):
    case_name = representative_df.loc[
        representative_df["Sample_ID"] == sample_id, "Case"
    ].iloc[0]
    
    shap_vector = representative_shap_values[case_number]
    
    case_lime_matrix = np.vstack([
        vector for key, vector in lime_vectors.items() if key[0] == case_name
    ])
    
    case_table = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Value": shap_vector,
        "Mean_ABS_SHAP": np.abs(shap_vector),
        "Mean_LIME_Value": np.mean(case_lime_matrix, axis=0),
        "SD_LIME": np.std(case_lime_matrix, axis=0, ddof=1)
    })
    
    case_table["SHAP_Rank"] = case_table["Mean_ABS_SHAP"].rank(
        ascending=False, method="min"
    ).astype(int)
    
    case_table["LIME_Rank"] = case_table["Mean_LIME_Value"].abs().rank(
        ascending=False, method="min"
    ).astype(int)
    
    safe_case_name = re.sub(r"[^A-Za-z0-9_]+", "_", case_name)
    case_table.to_csv(
        SAVE_DIR / "LIME" / f"{safe_case_name}_SHAP_LIME_comparison.csv",
        index=False
    )


# ================================================================
# 41. LIME REPRESENTATIVE CASE PLOTS
# ================================================================

for case_number, sample_id in enumerate(representative_sample_ids):
    case_name = representative_df.loc[
        representative_df["Sample_ID"] == sample_id, "Case"
    ].iloc[0]
    
    shap_vector = representative_shap_values[case_number]
    
    case_lime_matrix = np.vstack([
        vector for key, vector in lime_vectors.items() if key[0] == case_name
    ])
    mean_lime = np.mean(case_lime_matrix, axis=0)
    
    top_indices = np.argsort(np.abs(shap_vector))[-10:][::-1]
    plot_features = [feature_names[i] for i in top_indices]
    shap_plot_values = [shap_vector[i] for i in top_indices]
    lime_plot_values = [mean_lime[i] for i in top_indices]
    
    y_positions = np.arange(len(plot_features))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    axes[0].barh(y_positions, shap_plot_values)
    axes[0].set_yticks(y_positions)
    axes[0].set_yticklabels(plot_features)
    axes[0].invert_yaxis()
    axes[0].axvline(0, linestyle="--", color='gray', alpha=0.5)
    axes[0].set_xlabel("SHAP contribution")
    axes[0].set_title(f"{case_name}: SHAP")
    
    axes[1].barh(y_positions, lime_plot_values)
    axes[1].set_yticks(y_positions)
    axes[1].set_yticklabels(plot_features)
    axes[1].invert_yaxis()
    axes[1].axvline(0, linestyle="--", color='gray', alpha=0.5)
    axes[1].set_xlabel("Mean LIME contribution")
    axes[1].set_title(f"{case_name}: LIME")
    
    plt.suptitle("SHAP and LIME Local Explanation Comparison", fontsize=16, fontweight="bold")
    plt.tight_layout()
    
    safe_case_name = re.sub(r"[^A-Za-z0-9_]+", "_", case_name)
    plt.savefig(
        SAVE_DIR / "LIME" / f"{safe_case_name}_SHAP_vs_LIME.png",
        dpi=600, bbox_inches="tight"
    )
    plt.close()


# ================================================================
# 42. SAVE SHAP VALUES
# ================================================================

shap_data = {
    "shap_values": shap_values,
    "expected_value": expected_value,
    "feature_names": feature_names,
    "sample_ids": X_shap.index.tolist()
}

joblib.dump(shap_data, SAVE_DIR / "SHAP" / "test_set_SHAP_values.joblib")


# ================================================================
# 43. CREATE REVIEWER SUMMARY TABLE
# ================================================================

reviewer_summary = [
    {
        "Reviewer_Issue": "Feature dependence and correlated predictors",
        "Analysis_Performed": "Pearson correlation, Spearman correlation, VIF",
        "Status": "Completed",
        "Output": "Correlation tables, VIF_analysis.csv"
    },
    {
        "Reviewer_Issue": "SHAP interaction effects",
        "Analysis_Performed": "Full interaction matrix + reviewer-requested pairs",
        "Status": "Completed",
        "Output": "SHAP_interaction_matrix.csv, reviewer_requested_interactions.csv"
    },
    {
        "Reviewer_Issue": "Correlation-aware attribution",
        "Analysis_Performed": "Partition SHAP with correlation clustering",
        "Status": "Completed",
        "Output": "Partition_SHAP_correlation_aware.csv"
    },
    {
        "Reviewer_Issue": "Grouped attribution",
        "Analysis_Performed": "Grouped SHAP (combined model attribution) + grouped permutation",
        "Status": "Completed",
        "Output": "grouped_SHAP_attribution.csv, grouped_permutation_importance.csv"
    },
    {
        "Reviewer_Issue": "Conditional stratification of SHAP attribution",
        "Analysis_Performed": "SHAP attribution stratified by low/high levels of correlated variables",
        "Status": "Completed",
        "Output": "conditional_stratification_SHAP_attribution.csv"
    },
    {
        "Reviewer_Issue": "Feature ablation sensitivity",
        "Analysis_Performed": "Individual and group feature ablation with Δ metrics (using original preprocessing)",
        "Status": "Completed",
        "Output": "feature_ablation_sensitivity.csv"
    },
    {
        "Reviewer_Issue": "LIME representative cases (low/medium/high/ultra-high)",
        "Analysis_Performed": "Scientifically-defined strength ranges (<30, 30-50, 50-70, >70 MPa)",
        "Status": "Completed",
        "Output": "representative_strength_cases.csv"
    },
    {
        "Reviewer_Issue": "LIME multi-seed stability",
        "Analysis_Performed": "5 seeds (42, 52, 62, 72, 82)",
        "Status": "Completed",
        "Output": "LIME_stability_summary.csv"
    },
    {
        "Reviewer_Issue": "LIME perturbation settings",
        "Analysis_Performed": "3000 and 10000 samples, with/without local sampling",
        "Status": "Completed",
        "Output": "LIME_all_runs.csv"
    },
    {
        "Reviewer_Issue": "LIME CV and rank stability",
        "Analysis_Performed": "Coefficient of variation and feature rank stability",
        "Status": "Completed",
        "Output": "LIME_cv_rank_stability_summary.csv"
    },
    {
        "Reviewer_Issue": "SHAP-LIME quantitative consistency",
        "Analysis_Performed": "Spearman ρ, Top-5 overlap count, Jaccard similarity, sign agreement",
        "Status": "Completed",
        "Output": "SHAP_LIME_consistency_reviewer_table.csv"
    },
    {
        "Reviewer_Issue": "Direction verification (non-causal)",
        "Analysis_Performed": "Pearson/Spearman correlations, binned means",
        "Status": "Completed",
        "Output": "SHAP_direction_verification.csv"
    },
    {
        "Reviewer_Issue": "Causal interpretation limitation",
        "Analysis_Performed": "Explicit non-causal language throughout (e.g., 'associated with')",
        "Status": "Completed",
        "Output": "analysis_metadata.json, direction interpretations"
    },
    {
        "Reviewer_Issue": "SHAP workflow redesign",
        "Analysis_Performed": "Uses actual CatBoost model, not Extra Trees",
        "Status": "Completed",
        "Output": "Full script using catboost_model directly"
    }
]

reviewer_summary_df = pd.DataFrame(reviewer_summary)
reviewer_summary_df.to_csv(
    SAVE_DIR / "Tables" / "reviewer_comment_analysis_summary.csv",
    index=False
)

print("\n" + "="*80)
print("REVIEWER SUMMARY TABLE")
print("="*80)
print(reviewer_summary_df[["Reviewer_Issue", "Status"]].to_string(index=False))


# ================================================================
# 44. SAVE ANALYSIS METADATA
# ================================================================

metadata = {
    "model_type": "CatBoostRegressor",
    "target": TARGET_COLUMN,
    "interpretation_dataset": "Independent held-out test set",
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "features": feature_names,
    "random_state": RANDOM_STATE,
    "lime_seeds": LIME_SEEDS,
    "lime_num_samples": LIME_NUM_SAMPLES,
    "lime_sample_around_instance": LIME_SAMPLE_AROUND_INSTANCE,
    "lime_strength_categories": STRENGTH_CATEGORIES,
    "partition_shap_test_samples": partition_n,
    "reviewer_feature_groups": reviewer_groups,
    "interpretation_scope": (
        "Model-based attribution, not causal inference. "
        "All interpretations use non-causal language "
        "(e.g., 'associated with', 'contributes to', "
        "'model attribution'), and avoid mechanistic claims."
    ),
    "ablation_method": "Original preprocessing pipeline (imputer + CatBoost) re-fitted for each ablation"
}

with open(SAVE_DIR / "analysis_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)


# ================================================================
# 45. FINAL CONSOLE SUMMARY
# ================================================================

print("\n")
print("=" * 80)
print("REVIEWER-REQUESTED ANALYSES COMPLETED")
print("=" * 80)

print("\nMain outputs:")
analyses = [
    "1. Pearson/Spearman feature dependence",
    "2. VIF analysis",
    "3. Standard TreeSHAP (with additivity check)",
    "4. SHAP dependence plots",
    "5. SHAP interaction matrix",
    "6. Reviewer-requested interaction pairs (FULLY CORRECTED)",
    "7. Correlation-aware Partition SHAP",
    "8. Grouped SHAP (combined model attribution)",
    "9. Conditional stratification of SHAP attribution",
    "10. Grouped permutation importance",
    "11. Individual permutation importance",
    "12. Feature ablation (with original preprocessing)",
    "13. Reviewer-specified ablation scenarios",
    "14. Group feature ablation",
    "15. Four representative LIME strength cases (scientifically defined)",
    "16. Multi-seed LIME stability",
    "17. Multi-setting LIME stability",
    "18. LIME CV and rank stability",
    "19. SHAP-LIME consistency (reviewer table format)",
    "20. SHAP-LIME direction contradiction analysis",
    "21. Feature-level SHAP-LIME comparison",
    "22. Direction verification (non-causal language)",
    "23. Reviewer analysis summary"
]

for analysis in analyses:
    print(f"  {analysis}")

print(f"\nResults saved to: {SAVE_DIR}")


# ================================================================
# 46. SHAP-LIME CONSISTENCY SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("SHAP-LIME CONSISTENCY SUMMARY")
print("=" * 80)

if len(consistency_df) > 0:
    print("\nOverall SHAP-LIME agreement across all runs:")
    print(f"  Spearman ρ: {consistency_df['SHAP_LIME_Spearman'].mean():.4f} ± {consistency_df['SHAP_LIME_Spearman'].std():.4f}")
    print(f"  Top-5 overlap count: {consistency_df['Top5_Overlap_Count'].mean():.2f}/5")
    print(f"  Top-5 Jaccard: {consistency_df['Top5_Jaccard'].mean():.4f}")
    print(f"  Sign agreement: {consistency_df['Sign_Agreement'].mean():.4f}")
    
    print("\nPer-case summary:")
    for case_name in consistency_df["Case"].unique():
        case_data = consistency_df[consistency_df["Case"] == case_name]
        print(f"  {case_name}:")
        print(f"    Spearman ρ = {case_data['SHAP_LIME_Spearman'].mean():.4f}")
        print(f"    Top-5 overlap = {case_data['Top5_Overlap_Count'].mean():.2f}/5")
        print(f"    Sign agreement = {case_data['Sign_Agreement'].mean():.4f}")
    
    print("\nInterpretation:")
    print("  - Spearman correlations are low, indicating limited agreement in the detailed")
    print("    ordering of feature contributions between SHAP and LIME.")
    print("  - This reflects fundamental differences between the two explanation methods:")
    print("    * SHAP provides game-theoretically consistent Shapley values for all features")
    print("    * LIME optimizes local fidelity with sparse, interpretable explanations")
    print("  - Top-5 overlap shows substantial agreement on the most important features,")
    print("    with 4 out of 5 leading features shared across cases.")
    print("  - SHAP and LIME therefore provided partially complementary local explanations")
    print("    rather than equivalent feature-level attributions.")

print("\n" + "=" * 80)
print("SUMMARY OF FINAL CORRECTIONS")
print("=" * 80)
print("\n1. CORRECTED: SP = Superplasticizer (not Aspect Ratio)")
print("2. CORRECTED: Aspect Ratio = None (excluded from model)")
print("3. CORRECTED: Fiber_Geometry = ['Df', 'Lf']")
print("4. CORRECTED: 'OPC × Water' instead of 'Binder × Water'")
print("5. CORRECTED: Ablation uses original preprocessing pipeline (Option A)")
print("6. CORRECTED: Precise interpretation language for conditional stratification")
print("7. COMPLETE: Sections 1-46 fully implemented")
print("8. NON-CAUSAL: All interpretations use 'associated with' not causal language")
print("=" * 80)

CATBOOST SHAP + LIME REVIEWER VALIDATION (COMPLETE CORRECTED)
Output directory: D:\2026 Work\My Papers\1-3D-printed fiber-reinforced concrete\New Work\SHAP AND LIME

Loading saved CatBoost model...
CatBoost model loaded successfully.
Saved modelling results loaded successfully.

NOTE: Model uses numeric-input CatBoost (no cat_features specified).
      XAI analyses are consistent with this actual model.

Loading original dataset...

Features used by CatBoost:
  OPC
  Sand
  wtob
  FA
  GS
  SF
  SP
  HPMC
  W
  Fvol
  Df
  Lf

Original model split recovered:
Training samples: 180
Testing samples:  45

Model performance verification:
    Dataset        R2      RMSE       MAE
0  Training  0.944694  7.630541  5.924475
1   Testing  0.931851  9.056247  7.367220

Calculating feature dependence...

VIF analysis completed.

FEATURE MAPPING (FINAL CORRECTED)
  W (Water): W
  wtob (W/B): wtob
  SF (Silica Fume): SF
  Df (Fiber Diameter): Df
  Lf (Fiber Length): Lf
  AR (Aspect Ratio): None (excl

PartitionExplainer explainer: 46it [00:30,  1.09it/s]                        



Correlation-aware Partition SHAP completed.

Calculating conditional stratification of SHAP attribution...

Conditional stratification of SHAP attribution:
  The mean absolute SHAP magnitude of W was 1.15x higher in observations with high wtob than in observations with low wtob
  The mean absolute SHAP magnitude of SF was 1.17x higher in observations with high wtob than in observations with low wtob

Calculating grouped permutation importance...

Calculating individual permutation importance...

Starting feature-ablation sensitivity analysis...

Feature ablation completed with original preprocessing pipeline.

Selecting representative strength cases using scientifically-defined ranges...

Strength category availability in test set:
  Low: 2 samples - AVAILABLE
  Medium: 9 samples - AVAILABLE
  High: 2 samples - AVAILABLE
  Ultra-high: 32 samples - AVAILABLE

Selected representative cases:
      Case Strength_Category  Category_Median_MPa  Sample_ID  Actual_CS_MPa  Predicted_CS_MPa  Ab

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

...

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>